In [ ]:
import sys
from pathlib import Path
import os

In [ ]:
# Adding project root (parent of notebooks/) to PYTHONPATH
sys.path.append(str(Path.cwd().parent))

In [ ]:
from ingestion.loaders import extract_all_texts

extract_all_texts("../data/resume", "../data/resume_treated/resume_extract")

In [ ]:
assert "OPENAI_API_KEY" in os.environ, (
    "OPENAI_API_KEY is not set. "
    "Please follow the README instructions to configure it."
)

print("✅ OPENAI_API_KEY detected")


In [ ]:
from agents.structuring_agent import process_resumes
process_resumes(
    input_folder="../data/resume_treated/resume_extract",
    output_folder="../data/resume_treated/resume_extract_json"
)


In [ ]:
import json
from pathlib import Path

sample = next(Path("../data/resume_treated/resume_extract_json").glob("*.json"))

with open(sample, "r", encoding="utf-8") as f:
    data = json.load(f)

data


In [ ]:
from ingestion.formatters import process_json_folder

process_json_folder(
    "../data/resume_treated/resume_extract_json",
    "../data/resume_treated/resume_extract_text"
)


In [ ]:
from pathlib import Path
sample = next(Path("../data/resume_treated/resume_extract_text").glob("*.txt"))
sample.read_text(encoding="utf-8")[:1000]


In [ ]:
from ingestion.resume_index import build_resume_index

BACKEND = "faiss"   # or "chroma"

result = build_resume_index(
    input_folder="../data/resume_treated/resume_extract_text",
    backend=BACKEND,
    faiss_index_path="../data/resume_treated/resume_index.faiss",
    mapping_path="../data/resume_treated/resume_index_mapping.json",
    chroma_dir="../data/resume_treated/chroma_resume_db",
    chroma_collection="resumes",
)

result

In [ ]:
from ingestion.job_index import build_jobs_index

result = build_jobs_index(
    input_folder="../data/jobs",
    faiss_index_path="../data/jobs_index.faiss",
    mapping_path="../data/jobs_index_mapping.json"
)
result


In [ ]:
from app.matching import search_index, read_txt

job_text = read_txt("../data/jobs/sample_job.txt")

top_resumes = search_index(
    query_text=job_text,
    index_path="../data/resume_treated/resume_index.faiss",
    mapping_path="../data/resume_treated/resume_index_mapping.json",
    top_k=10
)
top_resumes


In [ ]:
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_resume_text(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_resume_text(
        filename=hit["filename"],
        base_dir="../data/resume_treated/resume_extract_text"
    )

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,
        resume_text=resume_text,
        top_k_rank=rank,
        client=client,
    )

    hit["llm_explanation"] = safe_json_parse(out["raw_json"])



In [ ]:
print("Score:", hit["score"])
print(hit["llm_explanation"]["explanation"])
print("Strengths:")
for s in hit["llm_explanation"]["strengths"]:
    print("-", s)
print("Gaps:")
for g in hit["llm_explanation"]["gaps"]:
    print("-", g)


In [ ]:
# =========================
# MODE 1 — JOB → TOP RESUMES
# =========================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"  # <- your folder with resume .txt

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_txt(hit["filename"], RESUME_DIR)

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,            # query = job
        resume_text=resume_text,      # candidate = resume
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[JOB → RESUMES] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


In [ ]:
resume_text = read_txt("../data/resume_treated/resume_extract_text/55.txt")

top_jobs = search_index(
    query_text=resume_text,
    index_path="../data/jobs_index.faiss",
    mapping_path="../data/jobs_index_mapping.json",
    top_k=10
)
top_jobs


In [ ]:
# ======================
# MODE 2 — RESUME → TOP JOBS
# ======================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

JOB_DIR = "../data/jobs"  # <- your folder with job .txt

for rank, hit in enumerate(top_jobs, start=1):
    job_candidate_text = read_txt(hit["filename"], JOB_DIR)

    out = explain_match_with_llm(
        mode="resume_to_jobs",
        similarity_score=hit["score"],
        resume_text=resume_text,           # query = resume
        job_text=job_candidate_text,       # candidate = job offer
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[RESUME → JOBS] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

def compatibility_score(job_text: str, resume_text: str, model_name="all-MiniLM-L6-v2") -> float:
    model = SentenceTransformer(model_name)
    emb = model.encode([job_text, resume_text], convert_to_numpy=True).astype("float32")
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    return float(np.dot(emb[0], emb[1]))  # cosine

score = compatibility_score(
    read_txt("../data/jobs/sample_job.txt"),
    read_txt("../data/resume_treated/resume_extract_text/55.txt")
)
score


In [ ]:
# =============================
# MODE 3 — PAIR COMPATIBILITY (RESUME ↔ JOB)
# =============================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"
JOB_DIR = "../data/jobs"

# Example inputs you choose (one resume + one job)
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_pair_text = read_txt(resume_filename, RESUME_DIR)
job_pair_text = read_txt(job_filename, JOB_DIR)

# score must be computed before (cosine similarity between embeddings)
# score = ...
out = explain_match_with_llm(
    mode="pair_compatibility",
    similarity_score=score,
    resume_text=resume_pair_text,
    job_text=job_pair_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

print("=" * 90)
print(f"[PAIR COMPATIBILITY] Resume: {resume_filename} | Job: {job_filename} | Score: {score:.3f}\n")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nDecision:", result.get("decision", "N/A"))


In [ ]:
# =========================================
# MODE 3 — PAIR COMPATIBILITY + CV IMPROVEMENT
# =========================================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"
JOB_DIR = "../data/jobs"

# Selected pair
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_text = read_txt(resume_filename, RESUME_DIR)
job_text = read_txt(job_filename, JOB_DIR)

# similarity_score must already be computed via embeddings
# score = cosine_similarity(...)
# Example:
# score = 0.78

out = explain_match_with_llm(
    mode="pair_compatibility_with_improvement",
    similarity_score=score,
    resume_text=resume_text,
    job_text=job_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

# --------------------
# DISPLAY
# --------------------
print("=" * 90)
print(
    f"[PAIR COMPATIBILITY]\n"
    f"Resume: {resume_filename}\n"
    f"Job: {job_filename}\n"
    f"Matching score: {score:.3f}\n"
)

print("Explanation:")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nCV Improvement Suggestions (targeted for this job):")
for i, rec in enumerate(result.get("cv_improvements", []), start=1):
    print(f"{i}. {rec}")

print("\nFinal Decision:", result.get("decision", "N/A"))


# LINKEDIN


In [ ]:
import pandas as pd

In [ ]:
df_job_posting = pd.read_csv("../data/linkedin_offers/linkedin_job_postings.csv")
df_skills = pd.read_csv("../data/linkedin_offers/job_skills.csv")
df_summary = pd.read_csv("../data/linkedin_offers/job_summary.csv")

In [ ]:
df_summary.dropna(inplace=True)
df_job_posting.dropna(inplace=True)
df_skills.dropna(inplace=True)

df_merged = df_job_posting.merge(df_skills, on="job_link").merge(df_summary, on="job_link")
df_merged.head()

In [ ]:
df_merged_reduced = df_merged[["job_link", "job_title", "company", "job_location", "search_city","job_type", "search_position", "job_skills", "job_summary"]]
df_merged_reduced.to_csv("linkedin_job_postings_cleaned.csv", index=False)